In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
from getpass import getpass
import subprocess

username = input("GitHub username: ")
token = getpass("GitHub Personal Access Token: ")

repo_url = "https://github.com/s-sana-sharifi/CAMELYON17-robustness.git"
repo_dir = "/content/CAMELYON17-robustness"

result = subprocess.run(
    [
        "git",
        "-c", f"http.extraHeader=Authorization: Basic "
               f"{__import__('base64').b64encode(f'{username}:{token}'.encode()).decode()}",
        "clone",
        repo_url,
        repo_dir,
    ],
    text=True,
    capture_output=True,
)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"git clone failed with exit code {result.returncode}")

print("Clone successful:", repo_dir)

GitHub username: s-sana-sharifi
GitHub Personal Access Token: ··········

Cloning into '/content/CAMELYON17-robustness'...

Clone successful: /content/CAMELYON17-robustness


In [5]:
from pathlib import Path

REPO_DIR = Path("/content/CAMELYON17-robustness")

print("Repo exists:", REPO_DIR.exists())
print("\nRepository files:")
for path in sorted(REPO_DIR.iterdir()):
    print(path.name)

Repo exists: True

Repository files:
.git
.gitignore
README.md
configs
notebooks
pyproject.toml
src
tests


In [6]:
from pathlib import Path

TAR_PATH = Path(
    "/content/drive/MyDrive/CAMELYON17-robustness/"
    "dataset/camelyon17_v1.0.tar"
)

print("Dataset exists:", TAR_PATH.exists())

if TAR_PATH.exists():
    print(f"Dataset size: {TAR_PATH.stat().st_size / 1024**3:.2f} GB")

Dataset exists: True
Dataset size: 10.68 GB


In [7]:
import tarfile
import pandas as pd

with tarfile.open(TAR_PATH, "r") as tar:
    with tar.extractfile("camelyon17_v1.0/metadata.csv") as f:
        metadata = pd.read_csv(f)

print("Metadata shape:", metadata.shape)
print("Patients:", metadata["patient"].nunique())
print("Centers:", metadata["center"].nunique())
print("Tumor labels:", sorted(metadata["tumor"].unique()))

Metadata shape: (455954, 9)
Patients: 43
Centers: 5
Tumor labels: [np.int64(0), np.int64(1)]


In [8]:
%cd /content/CAMELYON17-robustness

import sys
print("Working directory:", __import__("os").getcwd())
print("src exists:", __import__("os").path.isdir("src"))

/content/CAMELYON17-robustness
Working directory: /content/CAMELYON17-robustness
src exists: True


In [9]:
from src.data.sampling import sample_patches_by_patient

probe_metadata = sample_patches_by_patient(
    metadata,
    n_samples=500,
    seed=42,
)

print("Samples:", len(probe_metadata))
print("Patients:", probe_metadata["patient"].nunique())

print("\nSamples per patient:")
print(probe_metadata["patient"].value_counts().sort_index())

print("\nSamples per center:")
print(probe_metadata["center"].value_counts().sort_index())

Samples: 21500
Patients: 43

Samples per patient:
patient
4     500
9     500
10    500
12    500
15    500
16    500
17    500
20    500
21    500
22    500
24    500
34    500
36    500
38    500
39    500
40    500
41    500
42    500
44    500
45    500
46    500
48    500
51    500
52    500
60    500
61    500
62    500
64    500
66    500
67    500
68    500
72    500
73    500
75    500
80    500
81    500
86    500
87    500
88    500
89    500
92    500
96    500
99    500
Name: count, dtype: int64

Samples per center:
center
0    3500
1    4000
2    4500
3    5000
4    4500
Name: count, dtype: int64


In [10]:
import io
import tarfile

import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms


class TarPatchDataset(Dataset):
    def __init__(self, metadata, tar_path, transform=None):
        self.metadata = metadata.reset_index(drop=True)
        self.tar_path = tar_path
        self.transform = transform
        self._tar = None

    def _get_tar(self):
        if self._tar is None:
            self._tar = tarfile.open(self.tar_path, "r")
        return self._tar

    def _member_name(self, row):
        patient = int(row["patient"])
        node = int(row["node"])
        x = int(row["x_coord"])
        y = int(row["y_coord"])

        return (
            f"camelyon17_v1.0/patches/"
            f"patient_{patient:03d}_node_{node}/"
            f"patch_patient_{patient:03d}_node_{node}_"
            f"x_{x}_y_{y}.png"
        )

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, index):
        row = self.metadata.iloc[index]

        member_name = self._member_name(row)

        tar = self._get_tar()
        member = tar.getmember(member_name)

        file_obj = tar.extractfile(member)

        if file_obj is None:
            raise FileNotFoundError(
                f"Could not extract {member_name} from archive."
            )

        image = Image.open(io.BytesIO(file_obj.read())).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return {
            "image": image,
            "center": int(row["center"]),
            "patient": int(row["patient"]),
        }

In [11]:
probe_dataset = TarPatchDataset(
    probe_metadata,
    TAR_PATH,
)

sample = probe_dataset[0]

print("Image type:", type(sample["image"]))
print("Image size:", sample["image"].size)
print("Center:", sample["center"])
print("Patient:", sample["patient"])

Image type: <class 'PIL.Image.Image'>
Image size: (96, 96)
Center: 0
Patient: 4


In [12]:
from sklearn.model_selection import GroupKFold

print("GroupKFold is available.")

GroupKFold is available.


In [13]:
import numpy as np
from sklearn.model_selection import GroupKFold

X = np.zeros(len(probe_metadata))
groups = probe_metadata["patient"].to_numpy()

gkf = GroupKFold(n_splits=5)

folds = list(
    gkf.split(
        X=X,
        groups=groups,
    )
)

print("Number of folds:", len(folds))

for fold_id, (train_idx, val_idx) in enumerate(folds):
    train_patients = set(
        probe_metadata.iloc[train_idx]["patient"]
    )
    val_patients = set(
        probe_metadata.iloc[val_idx]["patient"]
    )

    overlap = train_patients & val_patients

    train_centers = sorted(
        probe_metadata.iloc[train_idx]["center"].unique()
    )
    val_centers = sorted(
        probe_metadata.iloc[val_idx]["center"].unique()
    )

    print(
        f"Fold {fold_id}: "
        f"train={len(train_idx)}, "
        f"val={len(val_idx)}, "
        f"train_patients={len(train_patients)}, "
        f"val_patients={len(val_patients)}, "
        f"patient_overlap={len(overlap)}"
    )

    print(f"  train centers: {train_centers}")
    print(f"  val centers:   {val_centers}")

Number of folds: 5
Fold 0: train=17000, val=4500, train_patients=34, val_patients=9, patient_overlap=0
  train centers: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
  val centers:   [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Fold 1: train=17000, val=4500, train_patients=34, val_patients=9, patient_overlap=0
  train centers: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
  val centers:   [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Fold 2: train=17000, val=4500, train_patients=34, val_patients=9, patient_overlap=0
  train centers: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
  val centers:   [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Fold 3: train=17500, val=4000, train_patients=35, val_patients=8, patient_overlap=0
  train centers: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
  val centers:   [np.int64(0), np.int64(1), np.int64(2), np.int6

In [14]:
import torch
from torchvision import transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [15]:
train_idx, val_idx = folds[0]

train_metadata = probe_metadata.iloc[train_idx].reset_index(drop=True)
val_metadata = probe_metadata.iloc[val_idx].reset_index(drop=True)

train_dataset = TarPatchDataset(
    train_metadata,
    TAR_PATH,
    transform=transform,
)

val_dataset = TarPatchDataset(
    val_metadata,
    TAR_PATH,
    transform=transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))

Train: 17000
Validation: 4500


In [16]:
batch = next(iter(train_loader))

print("Image shape:", batch["image"].shape)
print("Center shape:", batch["center"].shape)
print("Patient shape:", batch["patient"].shape)

print("Centers in batch:", batch["center"].tolist()[:10])

Image shape: torch.Size([64, 3, 96, 96])
Center shape: torch.Size([64])
Patient shape: torch.Size([64])
Centers in batch: [4, 0, 1, 0, 4, 2, 1, 3, 3, 3]


In [17]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Device: cuda
GPU: Tesla T4


In [18]:
import torch
import torch.nn as nn
from torchvision.models import ResNet18_Weights, resnet18

model = resnet18(weights=ResNet18_Weights.DEFAULT)

in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 5)

model = model.to(DEVICE)

print(model.fc)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 137MB/s]


Linear(in_features=512, out_features=5, bias=True)


In [19]:
images = batch["image"].to(DEVICE)

with torch.no_grad():
    logits = model(images)

print("Input:", images.shape)
print("Output:", logits.shape)

Input: torch.Size([64, 3, 96, 96])
Output: torch.Size([64, 5])


In [20]:
from pathlib import Path

SAMPLE_DIR = Path("/content/camelyon17_probe_sample")
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

print("Sample directory:", SAMPLE_DIR)

Sample directory: /content/camelyon17_probe_sample


In [21]:
import tarfile
import time

start = time.time()

with tarfile.open(TAR_PATH, "r") as tar:
    for i, (_, row) in enumerate(probe_metadata.iterrows(), start=1):

        patient = int(row["patient"])
        node = int(row["node"])
        x = int(row["x_coord"])
        y = int(row["y_coord"])

        member_name = (
            f"camelyon17_v1.0/patches/"
            f"patient_{patient:03d}_node_{node}/"
            f"patch_patient_{patient:03d}_node_{node}_"
            f"x_{x}_y_{y}.png"
        )

        output_path = (
            SAMPLE_DIR
            / f"patient_{patient:03d}_node_{node}_"
              f"x_{x}_y_{y}.png"
        )

        if not output_path.exists():
            member = tar.getmember(member_name)

            file_obj = tar.extractfile(member)

            if file_obj is None:
                raise FileNotFoundError(member_name)

            output_path.write_bytes(file_obj.read())

        if i % 1000 == 0:
            elapsed = time.time() - start
            print(
                f"{i:,}/{len(probe_metadata):,} "
                f"({i / len(probe_metadata) * 100:.1f}%) "
                f"- {elapsed:.1f}s"
            )

elapsed = time.time() - start

print(f"\nDone in {elapsed / 60:.1f} minutes.")
print(
    "Files:",
    len(list(SAMPLE_DIR.glob("*.png")))
)

1,000/21,500 (4.7%) - 112.0s
2,000/21,500 (9.3%) - 156.8s
3,000/21,500 (14.0%) - 200.1s
4,000/21,500 (18.6%) - 240.4s
5,000/21,500 (23.3%) - 278.3s
6,000/21,500 (27.9%) - 315.9s
7,000/21,500 (32.6%) - 352.6s
8,000/21,500 (37.2%) - 388.7s
9,000/21,500 (41.9%) - 424.2s
10,000/21,500 (46.5%) - 459.2s
11,000/21,500 (51.2%) - 491.7s
12,000/21,500 (55.8%) - 521.7s
13,000/21,500 (60.5%) - 549.1s
14,000/21,500 (65.1%) - 575.3s
15,000/21,500 (69.8%) - 600.6s
16,000/21,500 (74.4%) - 624.7s
17,000/21,500 (79.1%) - 643.3s
18,000/21,500 (83.7%) - 657.9s
19,000/21,500 (88.4%) - 670.2s
20,000/21,500 (93.0%) - 681.0s
21,000/21,500 (97.7%) - 688.1s

Done in 11.5 minutes.
Files: 21500


In [22]:
from pathlib import Path

SAMPLE_DIR = Path("/content/camelyon17_probe_sample")

files = list(SAMPLE_DIR.glob("*.png"))

print("Extracted files:", len(files))

assert len(files) == 21_500

print("✓ All 21,500 sampled patches are present.")

Extracted files: 21500
✓ All 21,500 sampled patches are present.


In [23]:
# Local filesystem dataset + patient-grouped folds

from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import GroupKFold

SAMPLE_DIR = Path("/content/camelyon17_probe_sample")

# Rebuild metadata if needed
# probe_metadata should already exist; this makes sure it is clean.
probe_metadata = probe_metadata.reset_index(drop=True).copy()

# Reconstruct local filenames from metadata
def make_local_filename(row):
    patient = int(row["patient"])
    node = int(row["node"])
    x = int(row["x_coord"])
    y = int(row["y_coord"])

    return (
        f"patient_{patient:03d}_node_{node}_"
        f"x_{x}_y_{y}.png"
    )

probe_metadata["filename"] = probe_metadata.apply(
    make_local_filename,
    axis=1,
)

# Verify every sampled image exists
missing = [
    name
    for name in probe_metadata["filename"]
    if not (SAMPLE_DIR / name).exists()
]

print("Missing files:", len(missing))

assert len(missing) == 0

class LocalPatchDataset(Dataset):
    def __init__(self, metadata, image_dir, transform=None):
        self.metadata = metadata.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, index):
        row = self.metadata.iloc[index]

        image_path = self.image_dir / row["filename"]

        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return {
            "image": image,
            "center": int(row["center"]),
            "patient": int(row["patient"]),
        }


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

# Patient-grouped 5-fold CV
X = np.zeros(len(probe_metadata))
groups = probe_metadata["patient"].to_numpy()

gkf = GroupKFold(n_splits=5)
folds = list(gkf.split(X=X, groups=groups))

train_idx, val_idx = folds[0]

train_metadata = probe_metadata.iloc[train_idx].reset_index(drop=True)
val_metadata = probe_metadata.iloc[val_idx].reset_index(drop=True)

print("Fold 0")
print("  train samples:", len(train_metadata))
print("  val samples:", len(val_metadata))
print("  train patients:", train_metadata["patient"].nunique())
print("  val patients:", val_metadata["patient"].nunique())

assert set(train_metadata["patient"]).isdisjoint(
    set(val_metadata["patient"])
)

train_dataset = LocalPatchDataset(
    train_metadata,
    SAMPLE_DIR,
    transform=transform,
)

val_dataset = LocalPatchDataset(
    val_metadata,
    SAMPLE_DIR,
    transform=transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

print("✓ Local dataset and patient-grouped fold ready.")

Missing files: 0
Fold 0
  train samples: 17000
  val samples: 4500
  train patients: 34
  val patients: 9
Train batches: 266
Val batches: 71
✓ Local dataset and patient-grouped fold ready.


In [27]:
import time
import torch
import torch.nn as nn
from torchvision.models import ResNet18_Weights, resnet18

device = torch.device("cuda")

model = resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 5)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4,
)

NUM_EPOCHS = 5

for epoch in range(NUM_EPOCHS):
    # -----------------
    # Training
    # -----------------
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    start = time.time()

    for batch in train_loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["center"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * labels.size(0)

        predictions = outputs.argmax(dim=1)
        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)

    train_loss /= train_total
    train_accuracy = train_correct / train_total

    # -----------------
    # Validation
    # -----------------
    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in val_loader:
            images = batch["image"].to(device, non_blocking=True)
            labels = batch["center"].to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * labels.size(0)

            predictions = outputs.argmax(dim=1)

            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)

            all_predictions.extend(
                predictions.cpu().numpy()
            )
            all_labels.extend(
                labels.cpu().numpy()
            )

    val_loss /= val_total
    val_accuracy = val_correct / val_total

    elapsed = time.time() - start

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"time={elapsed:.1f}s | "
        f"train_loss={train_loss:.4f} | "
        f"train_acc={train_accuracy:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_acc={val_accuracy:.4f}"
    )

Epoch 1/5 | time=30.3s | train_loss=0.2513 | train_acc=0.9013 | val_loss=0.2995 | val_acc=0.8869
Epoch 2/5 | time=28.1s | train_loss=0.0552 | train_acc=0.9806 | val_loss=0.2849 | val_acc=0.9011
Epoch 3/5 | time=27.9s | train_loss=0.0379 | train_acc=0.9862 | val_loss=0.2819 | val_acc=0.9067
Epoch 4/5 | time=28.4s | train_loss=0.0270 | train_acc=0.9906 | val_loss=0.4438 | val_acc=0.8864
Epoch 5/5 | time=26.8s | train_loss=0.0246 | train_acc=0.9912 | val_loss=0.2643 | val_acc=0.9118


In [28]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
)

accuracy = accuracy_score(
    all_labels,
    all_predictions,
)

balanced_accuracy = balanced_accuracy_score(
    all_labels,
    all_predictions,
)

cm = confusion_matrix(
    all_labels,
    all_predictions,
    labels=[0, 1, 2, 3, 4],
)

print(f"Accuracy: {accuracy:.4f}")
print(f"Balanced Accuracy: {balanced_accuracy:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nPer-center classification report:")
print(
    classification_report(
        all_labels,
        all_predictions,
        labels=[0, 1, 2, 3, 4],
        target_names=[
            "Center 0",
            "Center 1",
            "Center 2",
            "Center 3",
            "Center 4",
        ],
        digits=4,
        zero_division=0,
    )
)

Accuracy: 0.9118
Balanced Accuracy: 0.9126

Confusion Matrix:
[[460   2   0  38   0]
 [ 27 969   0   4   0]
 [  1   0 972   0  27]
 [143   5   0 851   1]
 [  0   0 145   4 851]]

Per-center classification report:
              precision    recall  f1-score   support

    Center 0     0.7290    0.9200    0.8134       500
    Center 1     0.9928    0.9690    0.9808      1000
    Center 2     0.8702    0.9720    0.9183      1000
    Center 3     0.9487    0.8510    0.8972      1000
    Center 4     0.9681    0.8510    0.9058      1000

    accuracy                         0.9118      4500
   macro avg     0.9018    0.9126    0.9031      4500
weighted avg     0.9210    0.9118    0.9131      4500



In [29]:
from sklearn.model_selection import GroupKFold

X = np.zeros(len(probe_metadata))
groups = probe_metadata["patient"].to_numpy()

gkf = GroupKFold(n_splits=5)

folds = list(
    gkf.split(
        X=X,
        groups=groups,
    )
)

for fold_id, (train_idx, val_idx) in enumerate(folds):
    train_patients = set(probe_metadata.iloc[train_idx]["patient"])
    val_patients = set(probe_metadata.iloc[val_idx]["patient"])

    assert train_patients.isdisjoint(val_patients)

    print(
        f"Fold {fold_id + 1}: "
        f"train={len(train_idx):,}, "
        f"val={len(val_idx):,}, "
        f"train_patients={len(train_patients)}, "
        f"val_patients={len(val_patients)}"
    )

Fold 1: train=17,000, val=4,500, train_patients=34, val_patients=9
Fold 2: train=17,000, val=4,500, train_patients=34, val_patients=9
Fold 3: train=17,000, val=4,500, train_patients=34, val_patients=9
Fold 4: train=17,500, val=4,000, train_patients=35, val_patients=8
Fold 5: train=17,500, val=4,000, train_patients=35, val_patients=8


In [30]:
import time
import numpy as np
import torch
import torch.nn as nn

from torchvision.models import ResNet18_Weights, resnet18
from sklearn.metrics import accuracy_score, balanced_accuracy_score


device = torch.device("cuda")

NUM_EPOCHS = 5
NUM_CLASSES = 5

fold_results = []

for fold_id, (train_idx, val_idx) in enumerate(folds, start=1):

    print(f"\n{'=' * 60}")
    print(f"FOLD {fold_id}/5")
    print(f"{'=' * 60}")

    train_metadata = (
        probe_metadata.iloc[train_idx]
        .reset_index(drop=True)
    )

    val_metadata = (
        probe_metadata.iloc[val_idx]
        .reset_index(drop=True)
    )

    # ---------------------------------------------------------
    # Datasets
    # ---------------------------------------------------------

    train_dataset = LocalPatchDataset(
        train_metadata,
        SAMPLE_DIR,
        transform=transform,
    )

    val_dataset = LocalPatchDataset(
        val_metadata,
        SAMPLE_DIR,
        transform=transform,
    )

    # ---------------------------------------------------------
    # DataLoaders
    # ---------------------------------------------------------

    train_loader = DataLoader(
        train_dataset,
        batch_size=64,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=64,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True,
    )

    # ---------------------------------------------------------
    # Fresh model for this fold
    # ---------------------------------------------------------

    model = resnet18(
        weights=ResNet18_Weights.DEFAULT
    )

    model.fc = nn.Linear(
        model.fc.in_features,
        NUM_CLASSES,
    )

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-4,
    )

    # ---------------------------------------------------------
    # Training
    # ---------------------------------------------------------

    for epoch in range(NUM_EPOCHS):

        model.train()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        start = time.time()

        for batch in train_loader:

            images = batch["image"].to(
                device,
                non_blocking=True,
            )

            labels = batch["center"].to(
                device,
                non_blocking=True,
            )

            optimizer.zero_grad(set_to_none=True)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels,
            )

            loss.backward()
            optimizer.step()

            train_loss += (
                loss.item() * labels.size(0)
            )

            predictions = outputs.argmax(dim=1)

            train_correct += (
                predictions == labels
            ).sum().item()

            train_total += labels.size(0)

        train_loss /= train_total
        train_accuracy = train_correct / train_total

        # -----------------------------------------------------
        # Validation
        # -----------------------------------------------------

        model.eval()

        val_loss = 0.0
        val_total = 0

        all_predictions = []
        all_labels = []

        with torch.no_grad():

            for batch in val_loader:

                images = batch["image"].to(
                    device,
                    non_blocking=True,
                )

                labels = batch["center"].to(
                    device,
                    non_blocking=True,
                )

                outputs = model(images)

                loss = criterion(
                    outputs,
                    labels,
                )

                val_loss += (
                    loss.item() * labels.size(0)
                )

                predictions = outputs.argmax(dim=1)

                val_total += labels.size(0)

                all_predictions.extend(
                    predictions.cpu().numpy()
                )

                all_labels.extend(
                    labels.cpu().numpy()
                )

        val_loss /= val_total

        val_accuracy = accuracy_score(
            all_labels,
            all_predictions,
        )

        val_balanced_accuracy = (
            balanced_accuracy_score(
                all_labels,
                all_predictions,
            )
        )

        elapsed = time.time() - start

        print(
            f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
            f"time={elapsed:.1f}s | "
            f"train_loss={train_loss:.4f} | "
            f"train_acc={train_accuracy:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_acc={val_accuracy:.4f} | "
            f"val_bal_acc={val_balanced_accuracy:.4f}"
        )

    # ---------------------------------------------------------
    # Store final fold result
    # ---------------------------------------------------------

    fold_results.append(
        {
            "fold": fold_id,
            "accuracy": val_accuracy,
            "balanced_accuracy": val_balanced_accuracy,
        }
    )

    print(
        f"\nFold {fold_id} final: "
        f"accuracy={val_accuracy:.4f}, "
        f"balanced_accuracy={val_balanced_accuracy:.4f}"
    )


# =============================================================
# Overall result
# =============================================================

accuracies = np.array(
    [r["accuracy"] for r in fold_results]
)

balanced_accuracies = np.array(
    [r["balanced_accuracy"] for r in fold_results]
)

print("\n" + "=" * 60)
print("5-FOLD RESULTS")
print("=" * 60)

for result in fold_results:
    print(
        f"Fold {result['fold']}: "
        f"accuracy={result['accuracy']:.4f}, "
        f"balanced_accuracy={result['balanced_accuracy']:.4f}"
    )

print("\nAccuracy:")
print(
    f"{accuracies.mean():.4f} ± "
    f"{accuracies.std(ddof=1):.4f}"
)

print("\nBalanced Accuracy:")
print(
    f"{balanced_accuracies.mean():.4f} ± "
    f"{balanced_accuracies.std(ddof=1):.4f}"
)


FOLD 1/5
Epoch 1/5 | time=31.6s | train_loss=0.2625 | train_acc=0.8979 | val_loss=0.3080 | val_acc=0.8860 | val_bal_acc=0.8776
Epoch 2/5 | time=27.1s | train_loss=0.0600 | train_acc=0.9782 | val_loss=0.2578 | val_acc=0.9080 | val_bal_acc=0.8958
Epoch 3/5 | time=27.1s | train_loss=0.0298 | train_acc=0.9894 | val_loss=0.2506 | val_acc=0.9224 | val_bal_acc=0.9178
Epoch 4/5 | time=28.6s | train_loss=0.0265 | train_acc=0.9914 | val_loss=0.4930 | val_acc=0.8720 | val_bal_acc=0.8712
Epoch 5/5 | time=28.3s | train_loss=0.0231 | train_acc=0.9918 | val_loss=0.4411 | val_acc=0.8729 | val_bal_acc=0.8668

Fold 1 final: accuracy=0.8729, balanced_accuracy=0.8668

FOLD 2/5
Epoch 1/5 | time=28.3s | train_loss=0.2487 | train_acc=0.9033 | val_loss=0.3034 | val_acc=0.8813 | val_bal_acc=0.8810
Epoch 2/5 | time=29.1s | train_loss=0.0494 | train_acc=0.9826 | val_loss=0.2945 | val_acc=0.8889 | val_bal_acc=0.8858
Epoch 3/5 | time=27.8s | train_loss=0.0306 | train_acc=0.9900 | val_loss=0.2908 | val_acc=0.9080 